# 12. XGBoost Training (4개 모델)

## 목적
4개 예측 타겟에 대해 XGBoost 모델 학습 및 평가

## 예측 타겟
| 모델 | 타겟 | 예측 시점 | 임상적 의미 |
|------|------|----------|------------|
| Mortality | death_next_24h | 24시간 내 사망 | 전반적 예후 |
| Ventilator | vent_next_12h | 12시간 내 인공호흡기 | 호흡 악화 |
| Vasopressor | pressor_next_12h | 12시간 내 승압제 | 순환 악화 |
| Composite | composite_next_24h | 24시간 내 악화 | 종합 악화 |

## 입력
- `train.csv`, `val.csv`, `test.csv`
- `feature_cols.json`
- `scale_pos_weights.json`

## 출력
- 학습된 모델 (`.json`)
- 성능 결과 (`results_*.json`, `results_*.csv`)
- 시각화 (ROC, PR curves)

## Step 3: Vital Signs 결측치 처리

* 대상: hr, rr, spo2, temp, sbp, dbp, mbp
    * 결측률: 1~4% (매우 낮음)

* 전략: FFill → Median
    * ICU에서 거의 hourly 모니터링 → 결측 = 센서 일시 탈착 수준
    * 직전 값으로 채우는 것이 임상적으로 합리적
    * 결측률 낮아서 복잡한 처리 불필요

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import xgboost as xgb
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 성능 평가 모듈
from metrics import (
    evaluate_model, print_summary, print_confusion_matrices,
    plot_roc_curves, plot_pr_curves, plot_combined_roc,
    save_results, plot_feature_importance
)

print("=== 02. XGBoost Training 시작 ===")
print(f"XGBoost version: {xgb.__version__}")

## 설정

In [ ]:
# ==============================================================================
# 설정
# ==============================================================================

# 데이터 경로 (11_model_prep에서 생성된 폴더)
DATA_DIR = '../../data-pipeline/data/processed/all_features'  # 또는 'top20', 'custom_features' 등
OUTPUT_DIR = '../models/xgboost'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 예측 타겟 정의
TARGETS = {
    'death': 'death_next_24h',        # 24시간 내 사망
    'vent': 'vent_next_12h',          # 12시간 내 인공호흡기
    'pressor': 'pressor_next_12h',    # 12시간 내 승압제
    'composite': 'composite_next_24h'  # 24시간 내 종합 악화
}

# XGBoost 기본 하이퍼파라미터
BASE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'aucpr'],
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

# Early Stopping
EARLY_STOPPING_ROUNDS = 50
N_ESTIMATORS = 500
VERSION = 'v2'
os.makedirs(f'{OUTPUT_DIR}/{VERSION}', exist_ok=True)

print(f"데이터 경로: {DATA_DIR}")
print(f"모델 저장 경로: {OUTPUT_DIR}")

## Step 1: 데이터 로드

In [ ]:
print("\nStep 1: 데이터 로드")

# CSV 로드
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"✓ Train: {len(train):,} rows")
print(f"✓ Val: {len(val):,} rows")
print(f"✓ Test: {len(test):,} rows")

# 피처 목록 로드
with open(os.path.join(DATA_DIR, 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)
print(f"✓ 피처: {len(feature_cols)}개")

# 클래스 가중치 로드
with open(os.path.join(DATA_DIR, 'scale_pos_weights.json'), 'r') as f:
    scale_pos_weights = json.load(f)
print(f"✓ 클래스 가중치 로드 완료")

In [ ]:
# X, y 분리
X_train = train[feature_cols]
X_val = val[feature_cols]
X_test = test[feature_cols]

print(f"\n피처 shape:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

# 레이블 분포 확인
print(f"\n=== 타겟 레이블 분포 (Train) ===")
for name, col in TARGETS.items():
    pos_rate = train[col].mean() * 100
    pos_count = train[col].sum()
    weight = scale_pos_weights.get(col, 1)
    print(f"  {name}: {pos_count:,} ({pos_rate:.2f}%) | scale_pos_weight={weight:.1f}")

## Step 2: 모델 학습 (4개 타겟)

### 클래스 불균형 대응
- `scale_pos_weight`: Positive 클래스에 가중치 부여
- Early Stopping: Validation AUROC 기준

### 학습 전략
- 각 타겟별 별도 모델 학습
- Validation set으로 Early Stopping

In [ ]:
print("\nStep 2: 모델 학습")

models = {}
training_history = {}

for name, target_col in TARGETS.items():
    print(f"\n{'='*50}")
    print(f"[{name.upper()}] Training: {target_col}")
    print(f"{'='*50}")
    
    # 레이블 추출
    y_train = train[target_col]
    y_val = val[target_col]
    
    # 클래스 가중치
    weight = scale_pos_weights.get(target_col, 1)
    print(f"  scale_pos_weight: {weight:.1f}")
    
    # 파라미터 설정
    params = BASE_PARAMS.copy()
    params['scale_pos_weight'] = weight
    
    # 모델 생성 및 학습
    model = xgb.XGBClassifier(
        **params,
        n_estimators=N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )
    
    # 학습
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    # 결과 저장
    models[name] = model
    best_iteration = model.best_iteration
    
    # 학습 히스토리
    results = model.evals_result()
    training_history[name] = results
    
    print(f"  ✓ Best iteration: {best_iteration}")
    print(f"  ✓ Train AUC: {results['validation_0']['auc'][best_iteration]:.4f}")
    print(f"  ✓ Val AUC: {results['validation_1']['auc'][best_iteration]:.4f}")

print(f"\n{'='*50}")
print("✓ 모든 모델 학습 완료!")
print(f"{'='*50}")

## Step 3: 모델 평가 (Test Set)

In [ ]:
# 타겟별 최적 Threshold 설정
THRESHOLDS = {
    'death': 0.10, 
    'vent': 0.10,  
    'pressor': 0.10,
    'composite': 0.15
}

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.metrics import precision_score, recall_score, confusion_matrix

print("\nStep 3: 모델 평가 (Test Set)")

# 예측 및 평가
results = {}
y_true_dict = {}
y_prob_dict = {}

for name, col in TARGETS.items():
    y_true = test[col]
    y_prob = models[name].predict_proba(X_test)[:, 1]
    thresh = THRESHOLDS[name]
    y_pred = (y_prob >= thresh).astype(int)
    
    # 커브용 저장
    y_true_dict[name] = y_true
    y_prob_dict[name] = y_prob
    
    # Metrics 계산
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    results[name] = {
        'threshold': thresh,
        'auroc': auroc,
        'auprc': auprc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }

# 결과 출력
print("=" * 80)
print(" XGBoost v2 Performance (Custom Thresholds)")
print("=" * 80)
print(f"{'Model':<12} {'Thresh':>6} {'AUROC':>8} {'AUPRC':>8} {'F1':>8} {'Prec':>8} {'Recall':>8} {'Spec':>8}")
print("-" * 80)
for name, r in results.items():
    print(f"{name:<12} {r['threshold']:>6.2f} {r['auroc']:>8.4f} {r['auprc']:>8.4f} "
          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f} {r['specificity']:>8.4f}")
print("-" * 80)

In [ ]:
# TP/FP/FN 출력
print("\n[Detection Summary]")
print(f"{'Model':<12} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'Caught%':>10}")
print("-" * 60)
for name, r in results.items():
    total_pos = r['tp'] + r['fn']
    caught_pct = r['tp'] / total_pos * 100 if total_pos > 0 else 0
    print(f"{name:<12} {r['tp']:>8} {r['fp']:>8} {r['fn']:>8} {r['tn']:>8} {caught_pct:>9.1f}%")

In [ ]:
# Confusion Matrix 출력
print_confusion_matrices(results)

## Step 4: 시각화

In [ ]:
# ROC Curves (4개 모델)
plot_roc_curves(
    results, y_true_dict, y_prob_dict,
    save_path=os.path.join(f"{OUTPUT_DIR}/{VERSION}", 'roc_curves.png')
)

In [ ]:
# PR Curves (4개 모델)
plot_pr_curves(
    results, y_true_dict, y_prob_dict,
    save_path=os.path.join(f"{OUTPUT_DIR}/{VERSION}", 'pr_curves.png')
)

In [ ]:
# Combined ROC (단일 그래프)
plot_combined_roc(
    results, y_true_dict, y_prob_dict,
    save_path=os.path.join(f"{OUTPUT_DIR}/{VERSION}", 'roc_combined.png')
)

## Step 5: Feature Importance

In [ ]:
# Composite 모델 피처 중요도 (대표)
importance_dict = plot_feature_importance(
    models['composite'], 
    feature_cols, 
    top_n=20,
    save_path=os.path.join(f"{OUTPUT_DIR}/{VERSION}", 'feature_importance_composite.png'),
    title='Feature Importance (Composite Model)'
)

In [ ]:
# 4개 모델 피처 중요도 비교
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

titles = {
    'death': 'Mortality (24h)',
    'vent': 'Ventilator (12h)',
    'pressor': 'Vasopressor (12h)',
    'composite': 'Composite (24h)'
}

for idx, (name, model) in enumerate(models.items()):
    ax = axes[idx]
    
    # 피처 중요도
    importance = model.feature_importances_
    indices = np.argsort(importance)[::-1][:15]
    
    y_pos = np.arange(len(indices))
    ax.barh(y_pos, [importance[i] for i in indices[::-1]], color='steelblue')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_cols[i] for i in indices[::-1]], fontsize=9)
    ax.set_xlabel('Importance')
    ax.set_title(titles[name], fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(f"{OUTPUT_DIR}/{VERSION}", 'feature_importance_all.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Feature importance (all models) saved")

## Step 6: 모델 저장

In [ ]:
print("\nStep 6: 모델 저장")

import pickle

# 시간대별 디렉토리 생성
time_windows = {
    'death': '24h',
    'vent': '12h', 
    'pressor': '12h',
    'composite': '24h'
}

# 결과 저장 전에 numpy 타입 변환
def convert_to_native(obj):
    """numpy 타입을 Python 기본 타입으로 변환"""
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, (np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

# 각 모델 저장
for name, model in models.items():
    time_window = time_windows[name]
    
    # 경로: xgboost/v1/24h/death.pkl
    model_dir = os.path.join(OUTPUT_DIR, VERSION, time_window)
    os.makedirs(model_dir, exist_ok=True)
    
    model_path = os.path.join(model_dir, f'{name}.pkl')
    
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    
    print(f"  ✓ {name}: {model_path}")


# 결과 저장 (버전 폴더에)
results_dir = os.path.join(OUTPUT_DIR, VERSION)

# 변환 후 저장
results_native = convert_to_native(results)
save_results(results_native, results_dir, prefix='xgboost_')

In [ ]:
# 학습 설정 저장 (thresholds 추가)
config = {
    'model_type': 'XGBoost',
    'version': VERSION,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data_dir': DATA_DIR,
    'n_features': len(feature_cols),
    'feature_cols': feature_cols,
    'targets': TARGETS,
    'thresholds': THRESHOLDS,  # 추가
    'params': BASE_PARAMS,
    'n_estimators': N_ESTIMATORS,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'best_iterations': {name: model.best_iteration for name, model in models.items()},
    'results': results
}

# configs/xgboost/v2/ 경로에 저장
config_dir = os.path.join('../configs/xgboost', VERSION)
os.makedirs(config_dir, exist_ok=True)

config_path = os.path.join(config_dir, 'training_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"\n✓ Training config saved: {config_path}")

## Step 7: 최종 요약

In [ ]:
print("\n" + "="*70)
print(" 02. XGBoost Training 완료")
print("="*70)

print(f"\n📊 성능 요약 (Test Set):")
print(f"{'Model':<12} {'AUROC':>10} {'AUPRC':>10}")
print("-" * 35)
for name, metrics in results.items():
    print(f"{name:<12} {metrics['auroc']:>10.4f} {metrics['auprc']:>10.4f}")

avg_auroc = np.mean([m['auroc'] for m in results.values()])
avg_auprc = np.mean([m['auprc'] for m in results.values()])
print("-" * 35)
print(f"{'Average':<12} {avg_auroc:>10.4f} {avg_auprc:>10.4f}")

print(f"\n📁 저장된 파일:")
print(f"  - 모델: {OUTPUT_DIR}/{VERSION}/12h|24h/*.pkl")
print(f"  - 결과: {OUTPUT_DIR}/{VERSION}/xgboost_results_*.json")
print(f"  - 설정: ../configs/xgboost/{VERSION}/training_config.json")
print(f"  - 그래프: {OUTPUT_DIR}/*.png")

---
## 부록: 모델 로드 및 예측 코드

```python
import xgboost as xgb
import pandas as pd
import json

# 모델 로드
MODEL_DIR = '../models/xgboost'
DATA_DIR = '../data/processed/all_features'

model = xgb.XGBClassifier()
model.load_model(f'{MODEL_DIR}/xgb_composite.json')

# 피처 목록 로드
with open(f'{DATA_DIR}/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

# 새 데이터 예측
# new_data = pd.read_csv('new_patient.csv')
# X_new = new_data[feature_cols]
# y_prob = model.predict_proba(X_new)[:, 1]
```